# 🔗 4 モデル連歌リレー — 12 句

**遊びの実験**: 連歌 (linked verse) は中世日本の集団詩。1 人が 5-7-5、次の人が 7-7、次が 5-7-5 … を交互に継いでいく形式。

ここでは 4 モデルを順に巡回させて **12 句の連歌**を一緒に編む。各モデルは「**直前の句しか見ない**」という連歌の精神に倣う (発句のテーマだけは共有)。

**観察ポイント**:
- 🎨 何句目で連想が突然飛躍するか
- 🌀 飛躍した後も意味は緩く繋がり続けるか
- 🎭 各モデルの作風 (派手 vs 渋い、抽象 vs 具体) が表れるか
- 🔗 `03_haiku_battle.ipynb` (並列+審査) と対をなす **逐次協創**の実験


## 1. セットアップ

In [ ]:
import os, re, random, time
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI(
    base_url="https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1",
    api_key=os.environ.get("LLMJP_API_KEY", "dummy"),
    timeout=300.0,
)

def chat(model, prompt, system="日本語で。", max_tokens=3000, temperature=0.8,
         max_retries=2):
    """内部 streaming 1 ショット。
    - thinking モデルでも reasoning を落とさない
    - stream 途中で中断したら自動リトライ (LLM-jp 8b/32b thinking で時々発生)
    - 全リトライ失敗時は、最後に集めた content または reasoning を返す
    """
    sys_msg = system + "\n\n/no_think"
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt}]
    extra = {"chat_template_kwargs": {"enable_thinking": False}}
    def _create(use_extra):
        kwargs = dict(model=model, messages=msgs, max_tokens=max_tokens,
                      temperature=temperature, stream=True)
        if use_extra:
            kwargs["extra_body"] = extra
        return client.chat.completions.create(**kwargs)
    last_reasoning = ""
    for attempt_i in range(max_retries + 1):
        try:
            try:
                stream = _create(True)
            except Exception:
                stream = _create(False)
        except Exception as e:
            if attempt_i < max_retries:
                wait = 2.0 ** attempt_i + random.uniform(0, 0.5)
                print(f"\n  ⚠️  create failed ({type(e).__name__}); retry {attempt_i+1}/{max_retries} after {wait:.1f}s",
                      flush=True)
                time.sleep(wait)
                continue
            return ""
        content, reasoning = [], []
        interrupted = None
        try:
            for chunk in stream:
                if not chunk.choices: continue
                d = chunk.choices[0].delta
                c = getattr(d, "content", None)
                if c: content.append(c)
                for f in ("reasoning_content", "reasoning"):
                    v = getattr(d, f, None)
                    if v: reasoning.append(v); break
        except Exception as e:
            interrupted = type(e).__name__
        text = "".join(content).strip()
        last_reasoning = "".join(reasoning).strip()
        if text:
            return text
        if not interrupted:
            return last_reasoning  # 正常終了で content 空 → reasoning を採用
        # 中断 + content 空 → リトライ
        if attempt_i < max_retries:
            wait = 2.0 ** attempt_i + random.uniform(0, 0.5)
            print(f"\n  ⚠️  stream interrupted ({interrupted}); retry {attempt_i+1}/{max_retries} after {wait:.1f}s",
                  flush=True)
            time.sleep(wait)
            continue
        print(f"\n  ⚠️  stream interrupted ({interrupted}); all retries exhausted",
              flush=True)
        return last_reasoning
    return last_reasoning

ALL = [m.id for m in client.models.list().data]
def pick(s):
    for m in ALL:
        if s.lower() in m.lower(): return m
    raise RuntimeError(f"no model matching {s!r}")

VOICES = {
    "🌸 LLM-jp 8b":  pick("llm-jp-4-8b"),
    "🗻 LLM-jp 32b": pick("llm-jp-4-32b"),
    "🐉 Qwen 27b":   pick("qwen"),
    "💎 Gemma 31b":  pick("gemma"),
}
print("4 モデル準備完了:")
for n, m in VOICES.items():
    print(f"  {n:18s} → {m}")


## 2. 連歌リレー本体 — 12 句

In [ ]:
THEME = "山の研究室、季節は秋"
N_VERSES = 12

def short_form(i):
    return "5-7-5" if i % 2 == 0 else "7-7"

def prompt_for(i, prev=None, theme=None):
    if i == 0:
        return (f"テーマ『{theme}』で連歌の発句 (5-7-5) を 1 句詠んでください。\n"
                "本文のみ、1 行で。説明や引用符は不要。")
    return (f"以下は連歌の途中の句です。次に来るべき {short_form(i)} の句を 1 句、\n"
            "前句の余韻を受けつつ意味を緩やかに転じて詠んでください。\n"
            "本文のみ、1 行で。説明や引用符は不要。\n\n"
            f"前句: {prev}")

def extract_verse(raw):
    lines = [l.strip(" 　\t-・*「」『』\"\'") for l in raw.strip().splitlines()]
    lines = [l for l in lines if l and not l.startswith("#")]
    if not lines: return "(no output)"
    candidates = [l for l in lines if 6 <= len(l) <= 50]
    return candidates[-1] if candidates else lines[-1]

voice_list = list(VOICES.items())
chain = []
prev = None
print(f"🎴 テーマ: 「{THEME}」\n")
for i in range(N_VERSES):
    voice_name, model = voice_list[i % len(voice_list)]
    pmt = prompt_for(i, prev=prev, theme=THEME)
    raw = chat(model, pmt, temperature=0.95, max_tokens=2500)
    verse = extract_verse(raw)
    chain.append({"i": i, "voice": voice_name, "form": short_form(i), "verse": verse})
    print(f"  {i+1:02d}. ({short_form(i):5s}) {voice_name:18s} → {verse}")
    prev = verse


## 3. 連歌全体を表示

In [ ]:
body = []
for x in chain:
    body.append(
        f"**{x['i']+1:02d}.** *({x['form']})* {x['voice']}  \n"
        f"  　{x['verse']}"
    )
display(Markdown(
    f"## 🎴 連歌『{THEME}』\n\n" + "\n\n".join(body)
))


## 4. 終わりに — 1 モデルに講評を頼む

In [ ]:
# Gemma に連歌全体の講評を 3 行で
critic_name, critic_model = list(VOICES.items())[-1]
critique_prompt = (
    f"以下はテーマ『{THEME}』で 4 つの AI モデルが交互に詠んだ連歌 12 句です。\n"
    "全体の流れと、特に印象的な転回点を 3 行以内で講評してください。\n\n" +
    "\n".join(f"{x['i']+1:02d}. {x['verse']}" for x in chain)
)
critique = chat(critic_model, critique_prompt, temperature=0.5, max_tokens=2000)
display(Markdown(f"### 🎙️ 講評 by {critic_name}\n\n{critique}"))


## おまけ

- テーマを `"都会の終電、雨上がりの匂い"` や `"夏祭りの後の静けさ"` に変えてみる
- `N_VERSES` を 24 や 36 に増やすと本格的になる (API 呼び出しが線形に増える点に注意)
- 連歌は本来「**前句の余韻を受けつつ、新しい光景に転じる**」のが醍醐味。AI でその感覚を再現できているか味わってみてください
